#### Import and Helpers

In [7]:
import sys, json
from pathlib import Path
import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve()))
from dataset.statistics import read_split, generate_report

In [8]:
def load_stats(ds_dir):
    fp = Path(ds_dir) / "statistic.json"
    if not fp.exists():
        generate_report(ds_dir)
    return json.loads(fp.read_text())

def match(meta, flt):
    for k, allowed in flt.items():
        v = meta.get(k)
        allowed = allowed if isinstance(allowed, (list, tuple, set)) else [allowed]
        hit = any(x in allowed for x in v) if isinstance(v, list) else v in allowed
        if not hit:
            return False
    return True

# Dataset Dashboard
1. Given dataset, display the data in prettier look (\n displayed as real newline etc)
2. Costumized filter that helps us fine the useful info

In [9]:
DATA_DIR  = "../../dataset/CPT/Nitin-10k-jac-functions"
SPLIT     = ["train", "valid"]
FORMAT    = "jsonl"


# Preview

In [10]:
N_DISPLAY = 10
FILTER    = {
    "class": ["graph"]
} # tags in the meta, can be a list

In [11]:
for split in SPLIT:
    print("=" * 80)
    print(f"SPLIT: {split}")
    print("=" * 80)
    shown = matched = 0
    for rec in read_split(DATA_DIR, split):
        if not match(rec.get("meta", {}), FILTER):
            continue
        matched += 1
        if shown < N_DISPLAY:
            shown += 1
            print(f"\n--- [{split} #{shown}] meta={rec.get('meta', {})} ---")
            print(rec.get("text", ""))
    print(f"\n>>> {split}: {matched} record(s) matched {FILTER}; displayed {shown}\n")


SPLIT: train

>>> train: 0 record(s) matched {'class': ['graph']}; displayed 0

SPLIT: valid

>>> valid: 0 record(s) matched {'class': ['graph']}; displayed 0



# Statistics

In [12]:
BAR   = ["class"] # columns you want to plot as a bar chart, in SFT "task_type" is worth seeing

In [ ]:
stats = load_stats(DATA_DIR)

n_rows = len(SPLIT)
n_cols = len(BAR) + 1  # +1 for length histogram
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)

for i, split in enumerate(SPLIT):
    props = stats[split]["property"]
    for j, field in enumerate(BAR):
        ax = axes[i][j]
        items = sorted(props[field].items(), key=lambda kv: -kv[1])
        labels, values = zip(*items) if items else ([], [])
        bars = ax.bar(labels, values, color="steelblue")
        ax.set_title(f"{split} — {field}  (n={stats[split]['count']})")
        ax.set_ylabel("count")
        ax.tick_params(axis="x", rotation=30)
        for b, v in zip(bars, values):
            ax.text(b.get_x() + b.get_width() / 2, v, str(v),
                    ha="center", va="bottom", fontsize=8)

    ax = axes[i][-1]
    hist = stats[split].get("histogram", {})
    labels, values = list(hist.keys()), list(hist.values())
    bars = ax.bar(labels, values, color="darkorange")
    length = stats[split]["length"]
    ax.set_title(f"{split} — length (chars)  avg={length['avg']}  med={length['median']}")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=30)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v, str(v),
                ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()
